# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bikram-Mondal3/flyrank-ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The ranked queue is intended to help an SEO or content team decide which content items deserve review first. Items are ranked using observed search demand and historical performance signals. Each recommendation includes reason codes so that a human can understand why the item received its priority.

The main reason codes are HIGH_DEMAND, HIGH_IMPRESSIONS, LOW_CTR, and REFRESH_PRIORITY. These codes describe observed signals and should be treated as decision-support rather than proof that a refresh will improve performance.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("content_refresh_anonymized.csv")

for col in ["search_volume", "impressions_90d", "clicks_90d"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["ctr_90d"] = (
    df["clicks_90d"] /
    df["impressions_90d"].replace(0, np.nan)
)

df = df.dropna(
    subset=["search_volume", "impressions_90d", "clicks_90d"]
).copy()

df["demand_score"] = df["search_volume"].rank(pct=True)

df["impression_score"] = (
    df["impressions_90d"].rank(pct=True)
)

df["ctr_opportunity"] = (
    1 - df["ctr_90d"].rank(pct=True)
)

df["baseline_score"] = (
    0.40 * df["demand_score"]
    + 0.30 * df["impression_score"]
    + 0.30 * df["ctr_opportunity"]
)

median_volume = df["search_volume"].median()
median_impressions = df["impressions_90d"].median()
median_ctr = df["ctr_90d"].median()

def reason_codes(row):
    reasons = []

    if row["search_volume"] >= median_volume:
        reasons.append("HIGH_DEMAND")

    if row["impressions_90d"] >= median_impressions:
        reasons.append("HIGH_IMPRESSIONS")

    if row["ctr_90d"] <= median_ctr:
        reasons.append("LOW_CTR")

    if len(reasons) >= 2:
        reasons.append("REFRESH_PRIORITY")

    return "|".join(reasons)

df["reason_code"] = df.apply(reason_codes, axis=1)

priority_threshold = df["baseline_score"].quantile(0.75)

df["action"] = np.where(
    df["baseline_score"] >= priority_threshold,
    "REVIEW_FOR_REFRESH",
    "LOWER_PRIORITY"
)

ranked = (
    df.sort_values(
        "baseline_score",
        ascending=False
    )
    .reset_index(drop=True)
)

ranked["rank"] = ranked.index + 1

display(
    ranked[
        [
            col for col in [
                "rank",
                "content_hash_id",
                "search_volume",
                "impressions_90d",
                "clicks_90d",
                "ctr_90d",
                "baseline_score",
                "action",
                "reason_code"
            ]
            if col in ranked.columns
        ]
    ].head(20)
)

,rank,search_volume,impressions_90d,clicks_90d,ctr_90d,baseline_score,action,reason_code
0,1,2900.0,16156,0,0.000000,0.909013,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
1,2,33100.0,12275,0,0.000000,0.905045,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
2,3,320.0,14519,0,0.000000,0.888501,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
3,4,210.0,16786,0,0.000000,0.885573,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
4,5,49500.0,6483,0,0.000000,0.882164,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
5,6,22200.0,6188,0,0.000000,0.879662,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
6,7,2900.0,5090,0,0.000000,0.867934,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
7,8,3600.0,4963,0,0.000000,0.867730,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
8,9,1900.0,5176,0,0.000000,0.867193,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
9,10,60500.0,4560,0,0.000000,0.867109,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

The intended users are SEO and content team members who need to prioritize pages for manual review. The queue can help identify content with relatively high search demand, substantial impressions, or relatively low click-through rate.

The output is intended for prioritization and decision-support. It should not be used as an automatic instruction to rewrite, delete, publish, or change content. The score does not establish causality and does not guarantee that refreshing a page will improve rankings, clicks, or impressions. It also should not be interpreted as a prediction of Google's ranking decisions.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Intended use:")
print("Prioritize content items for human review.")

print("\nTotal content items:", len(ranked))
print(
    "Review-for-refresh:",
    (ranked["action"] == "REVIEW_FOR_REFRESH").sum()
)
print(
    "Lower priority:",
    (ranked["action"] == "LOWER_PRIORITY").sum()
)

print("\nPriority percentage:")
print(
    f"{(ranked['action'] == 'REVIEW_FOR_REFRESH').mean() * 100:.2f}%"
)

Intended use:
Prioritize content items for human review.

Total content items: 27532
Review-for-refresh: 6883
Lower priority: 20649

Priority percentage:
25.00%


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on a recommendation, a human reviewer should verify the page context, the freshness and completeness of the underlying measurements, the current search intent, and whether the observed signals still represent a meaningful opportunity.

The system must not automatically rewrite, delete, publish, redirect, or substantially change content. It must also not automatically make decisions involving clients, private search queries, or other sensitive information. A high score is only a reason to investigate further, not an instruction to act.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked.head(20).copy()

top20["human_review_required"] = True

top20["no_go_automation"] = (
    "Do not automatically rewrite, delete, publish, "
    "redirect, or make major content changes."
)

review_cols = [
    col for col in [
        "rank",
        "content_hash_id",
        "baseline_score",
        "action",
        "reason_code",
        "human_review_required",
        "no_go_automation"
    ]
    if col in top20.columns
]

display(top20[review_cols])

print("Human review is required for all recommendations.")

,rank,baseline_score,action,reason_code,human_review_required,no_go_automation
0,1,0.909013,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,True,"Do not automatically rewrite, delete, publish,..."
1,2,0.905045,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,True,"Do not automatically rewrite, delete, publish,..."
2,3,0.888501,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,True,"Do not automatically rewrite, delete, publish,..."
3,4,0.885573,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,True,"Do not automatically rewrite, delete, publish,..."
4,5,0.882164,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,True,"Do not automatically rewrite, delete, publish,..."
5,6,0.879662,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,True,"Do not automatically rewrite, delete, publish,..."
6,7,0.867934,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,True,"Do not automatically rewrite, delete, publish,..."
7,8,0.867730,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,True,"Do not automatically rewrite, delete, publish,..."
8,9,0.867193,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,True,"Do not automatically rewrite, delete, publish,..."
9,10,0.867109,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,True,"Do not automatically rewrite, delete, publish,..."


Human review is required for all recommendations.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations should be monitored for changes in the underlying data and for changes in model or queue performance. Important triggers include substantial changes in feature distributions, increasing missingness, changes in the rate of high-priority recommendations, and declining agreement between recommendations and subsequent measured outcomes.

A retraining or re-evaluation cycle should be considered when the search environment or content population changes materially, when new historical data becomes available, or when measured performance on a held-out validation set declines. These triggers indicate that the current recommendations may no longer represent the observed data.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
monitoring = {
    "rows": len(ranked),
    "missing_ctr_percent": df["ctr_90d"].isna().mean() * 100,
    "median_search_volume": df["search_volume"].median(),
    "median_impressions_90d": df["impressions_90d"].median(),
    "median_clicks_90d": df["clicks_90d"].median(),
    "median_ctr_90d": df["ctr_90d"].median(),
    "review_priority_rate": (
        ranked["action"] == "REVIEW_FOR_REFRESH"
    ).mean() * 100
}

for key, value in monitoring.items():
    print(f"{key}: {value:.3f}" if isinstance(value, float) else f"{key}: {value}")

print("\nRecommended monitoring triggers:")
print("1. Feature distributions change materially.")
print("2. Missing-value rates increase.")
print("3. Priority recommendation rate changes materially.")
print("4. Held-out validation performance declines.")
print("5. New historical outcome data becomes available.")

rows: 27532
missing_ctr_percent: 0.000
median_search_volume: 10.000
median_impressions_90d: 905.000
median_clicks_90d: 1.000
median_ctr_90d: 0.001
review_priority_rate: 25.000

Recommended monitoring triggers:
1. Feature distributions change materially.
2. Missing-value rates increase.
3. Priority recommendation rate changes materially.
4. Held-out validation performance declines.
5. New historical outcome data becomes available.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked action queue and supporting summary are exported so that the results can be reused in the research paper. The exported files contain anonymized identifiers and measured search-performance fields only. No client names, URLs, or private search queries are included.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

queue_cols = [
    col for col in [
        "rank",
        "content_hash_id",
        "search_volume",
        "impressions_90d",
        "clicks_90d",
        "ctr_90d",
        "baseline_score",
        "action",
        "reason_code"
    ]
    if col in ranked.columns
]

queue = ranked[queue_cols].copy()

queue_path = output_dir / "content_action_playbook.csv"
queue.to_csv(queue_path, index=False)

summary = pd.DataFrame({
    "metric": [
        "total_content_items",
        "review_for_refresh",
        "lower_priority",
        "review_priority_rate"
    ],
    "value": [
        len(ranked),
        (ranked["action"] == "REVIEW_FOR_REFRESH").sum(),
        (ranked["action"] == "LOWER_PRIORITY").sum(),
        (ranked["action"] == "REVIEW_FOR_REFRESH").mean()
    ]
})

summary_path = output_dir / "action_playbook_summary.csv"
summary.to_csv(summary_path, index=False)

print(f"Queue saved to: {queue_path}")
print(f"Summary saved to: {summary_path}")

display(queue.head(20))
display(summary)

Queue saved to: work/outputs/content_action_playbook.csv
Summary saved to: work/outputs/action_playbook_summary.csv


,rank,search_volume,impressions_90d,clicks_90d,ctr_90d,baseline_score,action,reason_code
0,1,2900.0,16156,0,0.000000,0.909013,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
1,2,33100.0,12275,0,0.000000,0.905045,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
2,3,320.0,14519,0,0.000000,0.888501,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
3,4,210.0,16786,0,0.000000,0.885573,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
4,5,49500.0,6483,0,0.000000,0.882164,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
5,6,22200.0,6188,0,0.000000,0.879662,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
6,7,2900.0,5090,0,0.000000,0.867934,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
7,8,3600.0,4963,0,0.000000,0.867730,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
8,9,1900.0,5176,0,0.000000,0.867193,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...
9,10,60500.0,4560,0,0.000000,0.867109,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...


,metric,value
0,total_content_items,27532.00
1,review_for_refresh,6883.00
2,lower_priority,20649.00
3,review_priority_rate,0.25


In [7]:
print("SELF-CHECK")
print("=" * 50)

print("Sections completed: 5/5")
print("Rows in ranked queue:", len(ranked))
print("Output directory:", output_dir)
print("Queue exists:", queue_path.exists())
print("Summary exists:", summary_path.exists())

print("\nNo client names, URLs, or private queries are intentionally included.")
print("Recommendations are decision-support and require human review.")

SELF-CHECK
Sections completed: 5/5
Rows in ranked queue: 27532
Output directory: work/outputs
Queue exists: True
Summary exists: True

No client names, URLs, or private queries are intentionally included.
Recommendations are decision-support and require human review.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.